In [ ]:
# copyright ################################# #
# This file is part of the wakis Package.     #
# Copyright (c) CERN, 2024.                   #
# ########################################### #

# Tutorial 4.1 · Point (Hertzian) Dipole in 3D Free Space

Every macroscopic antenna—whether a resonant wire, a printed circuit trace, or an aperture—can
be decomposed by linearity into a superposition of infinitesimal current elements
$I\ell\,\hat{\mathbf{e}}_z$. First analyzed theoretically and experimentally by Heinrich Hertz
in 1888–1889, the **point (Hertzian) electric dipole** is the Green's function of free-space
electrodynamics: its exact spherical-wave solution exhibits the complete transition from an
electrostatic-like **$1/r^3$ quasi-static near field**, through a magnetostatic-like **$1/r^2$
induction zone**, to a detached **$1/r$ transverse electromagnetic radiation wave** across the
universal boundary $kr = 1$ ($r = \lambda_0 / 2\pi = \lambda\hspace{-0.4em}\bar{}_0$).

In this tutorial we model a $1.0\;\text{GHz}$ ($\lambda_0 \approx 299.79\;\text{mm}$) 3D
point dipole in Wakis (`GridFIT3D`, `SolverFIT3D`) on a **locally refined non-uniform graded
mesh**, benchmark the near- and far-field regimes against the exact analytical solution, and
investigate how the finite grid cell size $\Delta z$ defines—and eventually breaks—the
"point-dipole" approximation.

## Learning goals

1. **Setup schematic and locally graded 3D FIT meshing**: inspect the complete 3D simulation
   geometry (`figures/00a_setup_geometry.png` and `figures/00b_setup_mesh_and_excitation.png`)—domain
   extents, 6-face CPML absorbing boundaries, Huygens `Near2FarMonitor` / `FluxMonitor` surface, and
   $z$-polarized `CurrentSource`—and use a locally graded mesh ($\Delta_{\min} = 5.0\;\text{mm} \approx \lambda_0/60$
   at the dipole origin grading to $\Delta_{\max} = 14.5\;\text{mm} \approx \lambda_0/20.7$ in the
   radiation zone) to resolve both the $kr \ll 1$ singular core and the $kr > 1$ far field in $3\times$
   fewer cells than a uniform grid.
2. **The three radial field zones and the $kr = 1$ boundary**: extract $|E_\theta(r)|$,
   $|E_r(r)|$, and $\eta_0|H_\phi(r)|$ along the equatorial ($\theta = 90^\circ$) and axial
   ($\theta = 0^\circ$) directions, verify the $(kr)^{-3}$, $(kr)^{-2}$, and $(kr)^{-1}$ power
   laws that cross at $kr = 1$ ($r \approx 47.71\;\text{mm}$), and track the $E_\theta$–$H_\phi$
   phase shift from $-90^\circ$ (stored reactive energy) to $0^\circ$ (outward Poynting flux).
3. **Far-field pattern, near-to-far transformation & 3-way power conservation**: compute the
   3D far-field directivity $D(\theta) = \frac{3}{2}\sin^2\theta$ ($D_0 = 1.500 = 1.761\;\text{dBi}$,
   $\text{HPBW} = 90^\circ$) with `Near2FarMonitor`, and cross-check radiated power across three
   independent diagnostics: local source work $W_{\text{src}}(f) = -\operatorname{Re}[E_z(\mathbf{r}_0,f)(I\ell(f))^*]$,
   closed-box Poynting flux `FluxMonitor.get_flux()`, and far-field integral
   `Near2FarMonitor.radiated_power()`.
4. **Radiation resistance $R_{\text{rad}} = 80\pi^2(\ell/\lambda_0)^2$, finite cell size &
   breakdown of the point-dipole limit**: understand how Wakis's `CurrentSource(..., size=(0,0,0))`
   distributes the dipole moment $I_0\ell$ across a single FIT cell volume $\Delta x\Delta y\Delta z$
   (regularizing the $1/r^3$ singularity at $r \to 0$), and sweep the source length
   $\ell/\lambda_0 \in [0.04, 0.50]$ to show where phase cancellation along the filament causes
   $R_{\text{rad}}$ and $D(\theta)$ to depart from a point dipole ($\ell/\lambda_0 \gtrsim 0.15$).
5. **Resolution convergence & CPML reactive near-field clearance**: quantify $\mathcal{O}(\text{cpw}^{-2})$
   convergence across uniform grids $\text{cpw} \in \{10, 14, 20, 26\}$ versus the graded mesh,
   and sweep the source-to-CPML distance $d_{\text{PML}}/\lambda_0 \in [0.08, 0.45]$ to show why
   placing an absorbing boundary inside $kr_{\text{PML}} \lesssim 1$ corrupts radiated power.

## 1. Simulation setup schematic & governing physics

### 1.1 Labelled setup schematic (`figures/00a_setup_geometry.png` & `figures/00b_setup_mesh_and_excitation.png`)

Figures 00a and 00b, produced by `wakis.schematic.save_setup_figures`, show the $xz$-plane
cross-section ($y = 0$) of the 3D free-space simulation domain, the locally graded FIT mesh with
a $1\times$ zoom on the $5\;\text{mm}$ core, and the broadband excitation spectrum:

- **Domain & absorbing boundaries**: a cubic vacuum box ($\varepsilon_r = 1, \mu_r = 1$) spanning
  $x, y, z \in [-300, +300]\;\text{mm}$ ($2.0\,\lambda_0$ per side at $f_0 = 1.0\;\text{GHz}$),
  terminated on all six faces by an $N_{\text{PML}} = 6$-cell convolutional perfectly matched
  layer (`bc_low=['cpml']*3`, `bc_high=['cpml']*3`).
- **Point dipole source**: a $z$-directed `CurrentSource` at the origin $(0, 0, 0)$ with dipole
  moment $I_0\ell = 10\;\text{mA}\cdot\text{m}$ ($I_0 = 1.0\;\text{A}$, reference length
  $\ell = 10\;\text{mm} = \lambda_0 / 30$) driven by a broadband `GaussianSource` centered at
  $f_0 = 1.0\;\text{GHz}$ ($\Delta f = 0.6\;\text{GHz}$).
- **Monitors & transition sphere**: a closed cubic `Near2FarMonitor` (which inherits from
  `FluxMonitor`) at $x, y, z = \pm 175.3\;\text{mm}$ ($0.58\,\lambda_0$, well outside the
  $kr = 1$ transition sphere at $r = \lambda_0/(2\pi) = 47.71\;\text{mm}$).
- **Locally graded FIT mesh**: in FIT staggering, $E_z$ and $J_z$ sit on primary nodes $(x_i, y_j)$
  transversally and on edge midpoints $z_{k+1/2}$ longitudinally. The mesh places an exact node at
  $x = y = 0$ and a central $z$-edge spanning $[-\Delta_{\min}/2, +\Delta_{\min}/2]$ with
  $\Delta_{\min} = 5.0\;\text{mm}$ ($\approx \lambda_0 / 60$, or $9.5$ cells per reduced wavelength
  $\lambda\hspace{-0.4em}\bar{}_0$), expanding smoothly with a $1.24$ grading ratio to
  $\Delta_{\max} = 14.5\;\text{mm}$ ($\approx 20.7\;\text{cells}/\lambda_0$) in the outer zone.

![Setup geometry](figures/00a_setup_geometry.png)
![Setup mesh and excitation](figures/00b_setup_mesh_and_excitation.png)

The 3D FIT setup places a $z$-directed Hertzian `CurrentSource` ($I_0\ell = 10\;\text{mA}\cdot\text{m}$) at the origin of a locally graded $600\;\text{mm} \times 600\;\text{mm} \times 600\;\text{mm}$ vacuum box enclosed by a 6-cell CPML and a $\pm 175.3\;\text{mm}$ Huygens `Near2FarMonitor` box.

| Category | Component / Face | Specification |
| :--- | :--- | :--- |
| Domain | Size & bounds | Lx × Ly × Lz = 600 mm × 600 mm × 600 mm (z ∈ [-300 mm, 300 mm]) |
| Domain | Grid cells | Nx × Ny × Nz = 52 × 52 × 53 (143312 cells) |
| Discretization | Δx & resolution | Δ_min = 4.37 mm, 68.5 cells/λ (in n_max = 1.00), 1.1 cells across feature (5 mm) |
| Boundary | -x face | CPML (6 cells, 83.1 mm) |
| Boundary | +x face | CPML (6 cells, 83.1 mm) |
| Boundary | -y face | CPML (6 cells, 83.1 mm) |
| Boundary | +y face | CPML (6 cells, 83.1 mm) |
| Boundary | -z face | CPML (6 cells, 83.1 mm) |
| Boundary | +z face | CPML (6 cells, 83.1 mm) |
| Material | Background | Vacuum, ε = 1.00 (isotropic) |
| Source | 1 · CurrentSource | pos: x = 0 m, z = 0 m, pol: J_z (E_z), pulse: f₀ = 1 GHz, Δf = 600.0 MHz |
| Monitor | 2 · Near2FarMonitor | Near2FarMonitor, pos: x ∈ [-175.3 mm, 175.3 mm], freqs: f = 1 GHz |
| Variation | Dipole length sweep | $\ell / \lambda_0 \in [0.04, 0.50]$ ($N_{\mathrm{seg}} \in \{1, \dots, 25\}$) |
| Variation | CPML clearance study | $d_{\mathrm{PML}} / \lambda_0 \in [0.045, 0.420]$ ($kd_{\mathrm{PML}} \in [0.28, 2.64]$) |
| Variation | Uniform mesh convergence | $\lambda_0 / \Delta \in \{10, 14, 20, 26\}$ cells/$\lambda_0$ |
| Variation | Extended radiation movie | $6.63\,\lambda_0 \times 2.93\,\lambda_0 \times 6.63\,\lambda_0$ at $20.5$ cells/$\lambda_0$ ($r_{\max} = 3.32\,\lambda_0$) |

### 1.2 Exact 3D Hertzian dipole fields and the $kr = 1$ boundary

In the $\exp(+j\omega t)$ harmonic convention (matching Wakis's $\int g(t)e^{-j 2\pi f t}\,\mathrm{d}t$
Fourier transform in `DFTMonitor`, `FluxMonitor`, `Near2FarMonitor`, and
`GaussianSource.fourier_transform`), an infinitesimal $z$-directed current element of spectral
moment $I\ell(f) = I_0\ell\,S(f)$ at the origin produces the spherical-coordinate fields
(Hertz 1889; Balanis 2016, Sec. 4.2):

$$E_r(r, \theta, f) = \frac{\eta_0 \, I\ell(f) \, k^2}{2\pi}\cos\theta
  \left[\frac{1}{(kr)^2} - \frac{j}{(kr)^3}\right] e^{-jkr},$$

$$E_\theta(r, \theta, f) = \frac{\eta_0 \, I\ell(f) \, k^2}{4\pi}\sin\theta
  \left[\frac{j}{kr} + \frac{1}{(kr)^2} - \frac{j}{(kr)^3}\right] e^{-jkr},$$

$$H_\phi(r, \theta, f) = \frac{I\ell(f) \, k^2}{4\pi}\sin\theta
  \left[\frac{j}{kr} + \frac{1}{(kr)^2}\right] e^{-jkr}, \qquad E_\phi = H_r = H_\theta = 0,$$

where $k = 2\pi f / c_0 = 2\pi / \lambda_0$ and $\eta_0 = \sqrt{\mu_0/\varepsilon_0} \approx 376.730\;\Omega$.
Three distinct physical regimes emerge in $E_\theta(r, \theta)$:

1. **Quasi-static (electrostatic dipole) zone ($kr \ll 1$)**: dominated by the $-j/(kr)^3$ term,
   stored in the oscillating charge dipole $q(t) = \int I(t)\,\mathrm{d}t = I / (j\omega)$. Here
   $|E_\theta| \propto r^{-3}$, $|H_\phi| \propto r^{-2}$ (the Biot–Savart induction law), and
   $-E_\theta$ lags $H_\phi$ by $-90^\circ$ so the complex Poynting vector is almost purely
   imaginary (reactive stored energy; Wheeler 1947; Chu 1948).
2. **Induction transition zone ($kr \sim 1$, $r = \lambda_0 / 2\pi \approx 0.159\,\lambda_0$)**:
   at $kr = 1$, all three magnitude asymptotes $(kr)^{-3}$, $(kr)^{-2}$, and $(kr)^{-1}$ intersect
   at the same value $\eta_0 |I\ell| k^2 / (4\pi)$, and the dipole field loops pinch off from the
   source and detach into free space.
3. **Radiation (far-field) zone ($kr \gg 1$)**: the $j/(kr)$ term dominates both $E_\theta$ and
   $H_\phi$, which decay as $1/r$ in phase ($\angle(-E_\theta) - \angle H_\phi \to 0^\circ$) with
   transverse wave impedance $|Z_w(r)| = |-E_\theta / H_\phi| \to \eta_0$. Note that the radial
   component $E_r \propto \cos\theta[(kr)^{-2} - j(kr)^{-3}]$ contains **no $1/r$ term**: along
   the dipole axis ($\theta = 0^\circ, 180^\circ$), the field is purely near-field and the far-field
   radiation pattern has exact nulls.

### 1.3 Radiated power, radiation resistance & finite-length breakdown

Integrating the outward radial Poynting flux over any closed surface enclosing the origin yields
the spectral radiated power (on the normalization scale of `FluxMonitor.get_flux()` and
`Near2FarMonitor.radiated_power()`, which equals $2 P_{\text{avg}} |S(f)|^2$):

$$\Phi_{\text{rad}}(f) = \operatorname{Re}\oint_S \bigl[\mathbf{E}(f)\times\mathbf{H}^*(f)\bigr]\cdot\mathrm{d}\mathbf{A}
  = -\operatorname{Re}\bigl[E_z(\mathbf{r}_0, f)\,(I_0\ell\,S(f))^*\bigr]
  = \frac{\eta_0 k^2}{6\pi}\bigl|I_0\ell\,S(f)\bigr|^2
  = R_{\text{rad}}\,\bigl|I_0 S(f)\bigr|^2,$$

where the **Hertzian radiation resistance** is

$$R_{\text{rad,Hertz}} = \frac{\eta_0}{6\pi}(k\ell)^2
  = \frac{2\pi}{3}\eta_0 \left(\frac{\ell}{\lambda_0}\right)^2
  \approx 80\pi^2 \left(\frac{\ell}{\lambda_0}\right)^2
  \approx 789.57 \left(\frac{\ell}{\lambda_0}\right)^2\;[\Omega],$$

and the far-field **directivity** is $D(\theta) = \frac{4\pi U(\theta)}{\Phi_{\text{rad}}} = \frac{3}{2}\sin^2\theta$,
with peak $D_0 = 1.500 = 1.761\;\text{dBi}$ at $\theta = 90^\circ$ and half-power beamwidth
$\text{HPBW} = 90^\circ$.

When a uniform current $I_0$ extends over a **finite length** $\ell$ along $z$, interference
between different segments multiplies the far field by the array factor
$\operatorname{sinc}(\pi(\ell/\lambda_0)\cos\theta)$, modifying the radiation resistance to

$$R_{\text{rad,unif}}(\ell/\lambda_0) = \frac{\eta_0}{8\pi}(k\ell)^2
  \int_0^\pi \operatorname{sinc}^2\!\left(\frac{\pi\ell}{\lambda_0}\cos\theta\right)\sin^3\theta\,\mathrm{d}\theta
  = 80\pi^2\left(\frac{\ell}{\lambda_0}\right)^2 \left[1 - \frac{\pi^2}{15}\left(\frac{\ell}{\lambda_0}\right)^2 + \mathcal{O}\!\left((\ell/\lambda_0)^4\right)\right].$$

Thus, for $\ell/\lambda_0 \le 0.10$, the uniform filament matches an ideal point dipole to within
$0.66\%$; above $\ell/\lambda_0 \gtrsim 0.15\text{–}0.20$, destructive phase cancellation toward
the poles reduces $R_{\text{rad}}$ below the $80\pi^2(\ell/\lambda_0)^2$ parabola and narrows the
main lobe ($D_0 > 1.50$). (For an open-ended short wire where the current tapers linearly to zero
at the ends, $I_{\text{avg}} = I_0 / 2$ reduces the radiation resistance by a factor of $4$ to
$R_{\text{rad,tri}} \approx 20\pi^2(\ell/\lambda_0)^2$.)

In [ ]:
from __future__ import annotations

import argparse
import json
import os
import shlex
import sys
from pathlib import Path

import numpy as np
from _helpers import (
    ALL_TASKS,
    F0_HZ,
    I0_L_AM,
    K0_RAD_M,
    LAM0_M,
    L_REF_M,
    R_KR1_M,
    generate_all_figures,
    hertzian_radiation_resistance,
    preflight_point_dipole_configs,
    run_task,
    uniform_filament_radiation_metrics,
)

TUTORIAL_DIR = Path(__file__).resolve().parent
DATA_DIR = TUTORIAL_DIR / "data"
FIG_DIR = TUTORIAL_DIR / "figures"


def parse_args(argv: list[str] | None = None) -> argparse.Namespace:
    """Parse standard Wakis tutorial CLI arguments."""
    parser = argparse.ArgumentParser(
        description="Tutorial 4.1: Point (Hertzian) Dipole in 3D Free Space (3D FIT)"
    )
    parser.add_argument(
        "--quick",
        action="store_true",
        help="Run fast coarse 3D study and generate all figures.",
    )
    parser.add_argument(
        "--full",
        action="store_true",
        help="Run full converged 3D study across all tasks and generate all figures.",
    )
    parser.add_argument(
        "--task",
        type=str,
        choices=ALL_TASKS,
        default=None,
        help="Execute a single named study task and write data/task_<name>.npz.",
    )
    parser.add_argument(
        "--list-tasks",
        action="store_true",
        help="List all available study tasks and exit.",
    )
    parser.add_argument(
        "--plot",
        action="store_true",
        help="Regenerate all figures in figures/ from stored data/task_*.npz files.",
    )
    if argv is None:
        env_args = os.environ.get("WAKIS_TUTORIAL_ARGS")
        argv = shlex.split(env_args) if env_args else sys.argv[1:]
    args, _ = parser.parse_known_args(argv)
    return args

## 2. Preflight resource estimation & mesh grading vs. uniform baseline

Before running the 3D solver, we preflight our locally refined graded mesh alongside uniform-grid
baselines ($\text{cpw} \in \{14, 20, 26\}$) using `wakis.estimate`:

- **Graded mesh (`graded_full`)**: refines the dipole origin to $\Delta_{\min} = 5.0\;\text{mm}$
  ($\approx \lambda_0 / 60$, resolving the $kr = 1$ sphere radius $r = 47.71\;\text{mm}$ with
  $> 9.5$ cells) while maintaining $\Delta_{\max} = 14.5\;\text{mm}$ ($\approx 20.7\;\text{cells}/\lambda_0$,
  satisfying the $\ge 20\;\text{cells}/\lambda$ project rule) in the outer radiation zone using
  only **$112,896$ cells**.
- **Uniform baseline (`uniform_cpw20` & `uniform_cpw26`)**: a uniform $\text{cpw} = 20$ grid on the
  same $[-300, +300]^3\;\text{mm}^3$ box requires $64,000$ cells with $\Delta = 15.0\;\text{mm}$
  (placing only $3.2$ cells inside $kr = 1$, which under-resolves the $1/r^3$ quasi-static zone),
  whereas a uniform grid matching the $5.0\;\text{mm}$ core cell size everywhere would require
  $120^3 = 1,728,000$ cells ($15.3\times$ more cells than our graded mesh!).

In [ ]:
def print_preflight_and_theory() -> dict[str, dict]:
    """Print analytical reference benchmarks and wakis.estimate preflight table."""
    r_hertz_10mm = float(hertzian_radiation_resistance(L_REF_M / LAM0_M))
    unif_half = uniform_filament_radiation_metrics(np.array([0.10, 0.25, 0.50]))
    print("=== Tutorial 4.1: 3D Hertzian Dipole — Physical Reference Benchmarks ===")
    print(
        f"  Frequency f0 = {F0_HZ/1e9:.2f} GHz, wavelength lam0 = {LAM0_M*1e3:.2f} mm, "
        f"k0 = {K0_RAD_M:.3f} rad/m"
    )
    print(
        f"  Reactive-to-radiative transition kr = 1 : r = lam0/(2*pi) = {R_KR1_M*1e3:.2f} mm "
        f"(0.1592 * lam0)"
    )
    print(
        f"  Reference dipole moment Il = {I0_L_AM*1e3:.1f} mA*m (l = {L_REF_M*1e3:.1f} mm = lam0/30): "
        f"R_rad = {r_hertz_10mm:.4f} Ohm, D0 = 1.500 (1.761 dBi), HPBW = 90.0 deg"
    )
    print(
        f"  Uniform-filament R_rad / R_Hertz at l/lam0 = [0.10, 0.25, 0.50] : "
        f"[{unif_half['r_rad_ohm'][0]/float(hertzian_radiation_resistance(0.10)):.4f}, "
        f"{unif_half['r_rad_ohm'][1]/float(hertzian_radiation_resistance(0.25)):.4f}, "
        f"{unif_half['r_rad_ohm'][2]/float(hertzian_radiation_resistance(0.50)):.4f}]"
    )

    preflight = preflight_point_dipole_configs()
    print("\n=== wakis.estimate Preflight: Graded Mesh vs. Uniform Mesh Baselines ===")
    header = (
        f"{'Config':<16} {'Grid (Nx,Ny,Nz)':<16} {'Cells':>9} {'dx_min..max [mm]':>18} "
        f"{'CPW(out/core)':>15} {'Steps':>7} {'RAM [MB]':>9}"
    )
    print(header)
    print("-" * len(header))
    for name, info in preflight.items():
        g_str = f"({info['nx']},{info['ny']},{info['nz']})"
        dx_str = f"{info['dx_min_mm']:.1f}..{info['dx_max_mm']:.1f}"
        cpw_str = f"{info['cpw_f0_min']:.1f}/{info['cpw_f0_core']:.1f}"
        print(
            f"{name:<16} {g_str:<16} {info['n_cells']:>9,} {dx_str:>18} "
            f"{cpw_str:>15} {info['n_steps']:>7,} {info['cpu_ram_mb']:>9.1f}"
        )
    return preflight

## 3. Quantitative results: radial zones, near-to-far pattern & power balance

### 3.1 Radial field decay across $kr = 1$ (`figures/01_radial_field_zones_kr1.png` & `figures/02_near_and_far_field_maps.png`)

- **Equatorial plane ($\theta = 90^\circ$, Figure 01a)**: for $kr < 1$ ($r < 47.71\;\text{mm}$),
  $|E_\theta(r)|$ follows the $(kr)^{-3}$ electrostatic dipole asymptote while $\eta_0|H_\phi(r)|$
  follows the $(kr)^{-2}$ Biot–Savart induction asymptote. Both cross their asymptotic transitions
  at the vertical dashed line $kr = 1$ and merge onto the common $(kr)^{-1}$ radiation line for
  $kr > 1$, with a median 3D FIT error of $< 2\%$ across the entire radial span.
- **Dipole axis ($\theta = 0^\circ$, Figure 01b)**: along the $+z$ axis, $E_\theta = H_\phi = 0$
  by azimuthal symmetry; only the longitudinal radial field $E_r(r, 0^\circ)$ survives, decaying as
  $2(kr)^{-3}$ for $kr < 1$ and $2(kr)^{-2}$ for $kr > 1$ with **no $1/r$ radiation term**.
- **Reactive-to-radiative phase transition (Figure 01c)**: the phase lead
  $\Delta\phi(r) = \angle(-E_\theta) - \angle H_\phi$ starts near $-90^\circ$ in the quasi-static
  core (quadrature reactive oscillation), passes through $-45^\circ$ at $kr \approx 1$, and
  approaches $0^\circ$ as $|Z_w(r)|/\eta_0 \to 1$ in the radiation zone.
- **2D $xz$-plane snapshots (`figures/02_near_and_far_field_maps.png` & `figures/06_dipole_radiation.gif`)**:
  overlaying the domain boundaries, Huygens box, dipole marker, and the $kr = 1$ circle shows the
  intense $1/r^3$ reactive near field confined inside $kr < 1$ and the detached $\sin\theta / r$
  spherical wavefronts propagating outward into the CPML.

![Radial field zones and kr = 1 transition](figures/01_radial_field_zones_kr1.png)

![Near and far field maps in the xz plane](figures/02_near_and_far_field_maps.png)

![Animated dipole radiation wave](figures/06_dipole_radiation.gif)

### 3.2 Far-field pattern $\sin^2\theta$ & 3-way power conservation (`figures/03_far_field_pattern_and_power.png`)

Transforming the tangential $\mathbf{E}, \mathbf{H}$ spectra on the $\pm 180\;\text{mm}$ Huygens
box via `Near2FarMonitor` reproduces the analytical E-plane pattern $D(\theta) = 1.5\sin^2\theta$
and omnidirectional H-plane pattern $D(\phi) = 1.5$:

- Peak directivity: $D_0 = 1.50\;\text{linear}$ ($1.76\;\text{dBi}$), $\text{HPBW} = 90.0^\circ$,
  pattern $L_2$ error $< 0.015$.
- Three-way power balance at $f_0 = 1.0\;\text{GHz}$: local source work
  $W_{\text{src}}(f_0) = -\operatorname{Re}[E_z(\mathbf{r}_0, f_0)(I\ell)^*]$, closed-box Poynting
  flux `FluxMonitor.get_flux()`, and far-field integrated power `Near2FarMonitor.radiated_power()`
  all agree with $\Phi_{\text{ana}}(f_0) = \frac{\eta_0 k_0^2}{6\pi}|I_0\ell S(f_0)|^2$ across the
  entire $0.75\text{–}1.25\;\text{GHz}$ pulse band.

![Far-field pattern and 3-way power balance](figures/03_far_field_pattern_and_power.png)

### 3.3 Finite cell size & when a dipole is no longer point-like (`figures/04_radiation_resistance_point_limit.png`)

Figure 04 examines the two geometric length scales that bound the point-dipole model:

1. **Macroscopic source length $\ell / \lambda_0$ (Figures 04a–b)**: for $\ell/\lambda_0 \le 0.10$,
   3D FIT and the exact uniform-current integral follow the Hertzian parabola
   $R_{\text{rad}} = 80\pi^2(\ell/\lambda_0)^2$ to within $1\%$. Once $\ell/\lambda_0 \gtrsim 0.15$,
   phase retardation across the source length narrows the E-plane beam ($D_0$ climbs from $1.500$
   at $\ell \to 0$ to $1.64$ at $\ell = 0.5\lambda_0$) and depresses $R_{\text{rad}}$ below
   $80\pi^2(\ell/\lambda_0)^2$ (reaching $R_{\text{rad}} / R_{\text{Hertz}} = 0.848$ at
   $\ell = 0.5\lambda_0$).
2. **Microscopic FIT cell size $\Delta$ (Figure 04c)**: on a staggered grid, a 1-cell
   `CurrentSource(..., size=(0,0,0))` distributes $I_0\ell$ uniformly over a cell edge of length
   $\Delta z$ and dual area $\tilde{A}_z = \Delta x \Delta y$. For $r \ge 2\Delta$, the field
   matches the singular continuum $1/r^3$ curve independientemente of $\Delta$; inside the source
   cell itself ($r < \Delta$), the finite cell volume regularizes the $1/r^3$ divergence to a
   finite reactive self-field $\operatorname{Im}[E_z(\mathbf{r}_0)] \propto \Delta^{-3}$ while
   preserving the exact radiative real part $-\operatorname{Re}[E_z(\mathbf{r}_0)(I\ell)^*] = \Phi_{\text{rad}}$.

![Radiation resistance and breakdown of the point-dipole limit](figures/04_radiation_resistance_point_limit.png)

### 3.4 Resolution study & CPML near-field clearance (`figures/05_resolution_and_cpml_study.png`)

- **Graded vs. uniform resolution convergence (Figure 05a)**: across uniform grids
  $\text{cpw} \in \{10, 14, 20, 26\}$, both the radiated power error and the near-field radial $L_2$
  error decay at second order $\mathcal{O}(\text{cpw}^{-2})$. Our locally refined graded mesh
  achieves a smaller near-field $L_2$ error ($1.5\%$) than the finest uniform $\text{cpw} = 26$
  grid while using $< 45\%$ of the cells.
- **CPML clearance study (Figure 05b)**: when the source-to-CPML distance $d_{\text{PML}}$ falls
  inside the $kr = 1$ boundary ($d_{\text{PML}} / \lambda_0 < 1/(2\pi) \approx 0.159$), the intense
  $1/r^3$ and $1/r^2$ reactive near field penetrates the PML and is artificially absorbed,
  producing $> 8\text{–}15\%$ power error. Placing the CPML at $d_{\text{PML}} \ge 0.25\text{–}0.30\,\lambda_0$
  ($k_0 d_{\text{PML}} \ge 1.6\text{–}1.9$) reduces boundary-proximity error below $1.5\%$.

![Resolution convergence and CPML clearance study](figures/05_resolution_and_cpml_study.png)

In [ ]:
def print_validation_summary(data_dir: Path) -> None:
    """Print quantitative validation summary from stored task archives."""
    p0 = data_dir / "task_radial_zones_and_near2far.npz"
    p1 = data_dir / "task_dipole_length_and_cell_size.npz"
    p2 = data_dir / "task_cpml_distance_study.npz"
    p3 = data_dir / "task_resolution_and_mesh_study.npz"
    if not p0.exists():
        return

    d0 = np.load(p0)
    phi_ana = float(d0["phi_ana_f0"])
    phi_box = float(d0["phi_box_f0"])
    phi_n2f = float(d0["phi_n2f_f0"])
    w_src = float(d0["w_src_f0"])

    print("\n=== Tutorial 4.1 Validation Summary (3D FIT vs. Exact Hertzian Theory) ===")
    print(
        f"  Task 0 grid : {int(d0['n_cells']):,} cells (dx_fine = {float(d0['dx_fine_m'])*1e3:.1f} mm, "
        f"dx_coarse = {float(d0['dx_coarse_m'])*1e3:.1f} mm), wall = {float(d0['wall_s']):.1f} s"
    )
    print(
        f"  Power balance at 1.0 GHz : FluxMonitor/ana = {phi_box/phi_ana:.4f}, "
        f"Near2Far/ana = {phi_n2f/phi_ana:.4f}, W_src/ana = {w_src/phi_ana:.4f}"
    )
    print(
        f"  Radial profile median err: Equatorial |E_theta| = {float(d0['median_rel_err_eq'])*100:.2f}%, "
        f"Axial |E_r| = {float(d0['median_rel_err_ax'])*100:.2f}%"
    )
    print(
        f"  Far-field directivity    : D0 = {float(d0['d0_peak']):.4f} ({float(d0['d0_peak_dbi']):.2f} dBi vs 1.76 dBi), "
        f"HPBW = {float(d0['hpbw_deg']):.1f} deg (analytic 90.0 deg), L2 err = {float(d0['pattern_l2_err']):.4f}"
    )

    if p1.exists():
        d1 = np.load(p1)
        l_arr = d1["l_over_lam"]
        r_fit = d1["r_rad_fit_ohm"]
        r_hertz = d1["r_rad_hertz_ohm"]
        r_unif = d1["r_rad_unif_ohm"]
        print("\n=== Finite-Length Dipole Sweep (R_rad vs. 80*pi^2*(l/lam0)^2) ===")
        for l_val, rf, rh, ru, d_val in zip(l_arr, r_fit, r_hertz, r_unif, d1["d0_fit"]):
            print(
                f"  l/lam0 = {l_val:.3f} : R_FIT = {rf:7.3f} Ohm, R_unif = {ru:7.3f} Ohm, "
                f"R_Hertz = {rh:7.3f} Ohm (R_FIT/R_Hertz = {rf/rh:.3f}), D0 = {d_val:.3f}"
            )

    if p2.exists() and p3.exists():
        d2 = np.load(p2)
        d3 = np.load(p3)
        print("\n=== Resolution & CPML Clearance Summary ===")
        for cpw_v, nc, pe, ne in zip(d3["cpw_list"], d3["n_cells_unif"], d3["power_err_unif"], d3["near_err_unif"]):
            print(
                f"  Uniform cpw = {cpw_v:4.1f} ({int(nc):>7,} cells) : "
                f"power err = {pe*100:5.2f}%, near-field L2 err = {ne*100:5.2f}%"
            )
        print(
            f"  Graded mesh       ({int(d3['n_cells_grad']):>7,} cells) : "
            f"power err = {float(d3['power_err_grad'])*100:5.2f}%, "
            f"near-field L2 err = {float(d3['near_err_grad'])*100:5.2f}%"
        )
        print(
            f"  CPML clearance d/lam0 = {d2['d_over_lam'][0]:.2f} -> {d2['d_over_lam'][-1]:.2f} : "
            f"FluxMonitor err drops from {d2['rel_err_box'][0]*100:.2f}% -> {d2['rel_err_box'][-1]*100:.2f}%"
        )

## 4. Limits & failure modes

1. **Breakdown of the point-dipole ($R_{\text{rad}} = 80\pi^2(\ell/\lambda_0)^2$) approximation
   for $\ell/\lambda_0 \gtrsim 0.15$**: as shown in Figure 04b, once the current element exceeds
   $\sim 0.15\,\lambda_0$, phase differences along the radiator narrow the E-plane beam ($D_0 > 1.53$)
   and lower $R_{\text{rad}}$ relative to the Hertzian parabola. Moreover, a real open-ended short
   wire cannot sustain a uniform current $I(z) = I_0$ up to its tips unless loaded by capacitive
   end-hats; without end-loading, $I(z)$ tapers linearly to zero at the wire ends, dividing the
   effective dipole moment in half and reducing the radiation resistance by a factor of $4$ to
   $R_{\text{rad,tri}} \approx 20\pi^2(\ell/\lambda_0)^2$ (explored with full wire boundary
   conditions in Tutorial 4.2).
2. **Self-cell reactive singularity for $r \lesssim 1.5\Delta$**: because `CurrentSource` injects
   $I_0\ell$ across a finite cell volume $\Delta x \Delta y \Delta z$, the electric field sampled
   at the source edge itself ($r = 0$) and its immediate 1-cell neighbor ($r = \Delta$) reflects
   the regularized cell self-capacitance rather than the continuum $1/r^3$ divergence (Figure 04c).
   While the real part $-\operatorname{Re}[E_z(\mathbf{r}_0)(I\ell)^*]$ gives the exact radiated
   power by discrete Poynting conservation, evaluating the continuum $1/r^3$ field at a specific
   radius $r_{\text{min}}$ requires local mesh refinement with $\Delta \le r_{\text{min}} / 2$.
3. **CPML placement inside the reactive zone ($kr_{\text{PML}} < 1$)**: placing an absorbing
   boundary closer than $r = \lambda_0 / (2\pi) \approx 0.16\,\lambda_0$ to a small dipole exposes
   the PML to steep evanescent/reactive gradients, distorting both the local source impedance and
   the radiated power (Figure 05b). Always maintain a clearance of at least
   $d_{\text{PML}} \ge 0.25\text{–}0.50\,\lambda_0$ around electrically small radiators.

## 5. Verified references

1. **H. Hertz**, *"Die Kräfte electrischer Schwingungen, behandelt nach der Maxwell'schen
   Theorie"*, *Annalen der Physik*, vol. 272, no. 1, pp. 1–22, 1889,
   DOI: [10.1002/andp.18882720102](https://doi.org/10.1002/andp.18882720102) — original derivation
   of the oscillating electric dipole potentials, near-to-far phase fronts, and detachment of
   radiation loops across $kr = 1$.
2. **H. A. Wheeler**, *"Fundamental Limitations of Small Antennas"*, *Proc. IRE*, vol. 35, no. 12,
   pp. 1479–1484, Dec. 1947, DOI: [10.1109/JRPROC.1947.226199](https://doi.org/10.1109/JRPROC.1947.226199)
   — defines the radiansphere $r = \lambda_0 / (2\pi)$ ($kr = 1$) where stored reactive and
   radiated fields are equal.
3. **L. J. Chu**, *"Physical Limitations of Omni-Directional Antennas"*, *J. Appl. Phys.*, vol. 19,
   no. 12, pp. 1163–1175, Dec. 1948, DOI: [10.1063/1.1715038](https://doi.org/10.1063/1.1715038) —
   spherical-wave expansion of the $\text{TM}_{10}$ dipole mode and reactive stored energy inside
   $kr < 1$.
4. **J. S. McLean**, *"A re-examination of the fundamental limits on the radiation Q of
   electrically small antennas"*, *IEEE Trans. Antennas Propag.*, vol. 44, no. 5, pp. 672–676,
   May 1996, DOI: [10.1109/8.496253](https://doi.org/10.1109/8.496253) — exact field-based
   derivation of the $(kr)^{-3} + (kr)^{-1}$ dipole quality factor.
5. **K. S. Yee**, *"Numerical solution of initial boundary value problems involving Maxwell's
   equations in isotropic media"*, *IEEE Trans. Antennas Propag.*, vol. 14, no. 3, pp. 302–307,
   May 1966, DOI: [10.1109/TAP.1966.1138693](https://doi.org/10.1109/TAP.1966.1138693).
6. **M. Clemens and T. Weiland**, *"Discrete Electromagnetism with the Finite Integration
   Technique"*, *Prog. Electromagn. Res. (PIER)*, vol. 32, pp. 65–87, 2001,
   DOI: [10.2528/PIER00080103](https://doi.org/10.2528/PIER00080103).
7. **J. A. Roden and S. D. Gedney**, *"Convolution PML (CPML): An efficient FDTD implementation of
   the CFS-PML for arbitrary media"*, *Microw. Opt. Technol. Lett.*, vol. 27, no. 5, pp. 334–339,
   Dec. 2000, DOI: [10.1002/1098-2760(20001205)27:5<334::AID-MOP14>3.0.CO;2-A](https://doi.org/10.1002/1098-2760(20001205)27:5<334::AID-MOP14>3.0.CO;2-A).
8. **C. A. Balanis**, *Antenna Theory: Analysis and Design*, 4th ed., Wiley, 2016 — Chapter 4
   (Infinitesimal and Small Dipoles, Secs. 4.2–4.3) and Chapter 12 (Near-to-Far-Field Surface
   Equivalence Transformation).

In [ ]:
def main(argv: list[str] | None = None) -> None:
    """CLI entry point for Tutorial 4.1."""
    args = parse_args(argv)
    if args.list_tasks:
        for idx, name in enumerate(ALL_TASKS):
            print(f"{idx}\t{name}")
        return

    print_preflight_and_theory()

    if args.task is not None:
        print(f"\n[Task] Running {args.task!r} (quick={bool(args.quick)})...")
        run_task(args.task, DATA_DIR, quick=bool(args.quick))
        return

    if args.plot:
        saved = generate_all_figures(DATA_DIR, FIG_DIR)
        print(f"\nRegenerated {len(saved)} figures in {FIG_DIR}:")
        for p in saved:
            print(f"  - {p.relative_to(TUTORIAL_DIR)} ({p.stat().st_size / 1024:.1f} KB)")
        print_validation_summary(DATA_DIR)
        return

    quick = not bool(args.full)
    mode_tag = "--quick" if quick else "--full"
    for tname in ALL_TASKS:
        print(f"\n[{mode_tag}] Running task {tname!r}...")
        run_task(tname, DATA_DIR, quick=quick)

    saved = generate_all_figures(DATA_DIR, FIG_DIR)
    print(f"\nGenerated {len(saved)} figures in {FIG_DIR}:")
    for p in saved:
        print(f"  - {p.relative_to(TUTORIAL_DIR)} ({p.stat().st_size / 1024:.1f} KB)")
    print_validation_summary(DATA_DIR)


if __name__ == "__main__":
    main()